<div dir="rtl" style="text-align:right">
<h1 style="text-align:right"><bdi dir="ltr">Logits</bdi> را از مسیر واقعی مدل بازسازی کنید</h1>
<p style="text-align:right">درس 51 از 92 · یک جمله را تا امتیاز بعدی دنبال کنیم · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">45-trace</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Embedding</bdi>، موقعیت، بلوک‌ها و <bdi dir="ltr">Head</bdi> را با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace</code> همان مدل تطبیق دهید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: تمام</span> اجزای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> و تفاوت <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace</code> با یک محاسبهٔ جداگانه را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۴۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">آیا ورودی <bdi dir="ltr">Attention</bdi> بلوک اول همان <bdi dir="ltr">Token Embedding</bdi> خام است؟ کدام دو عملیات پیش از آن انجام می‌شوند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
model = MiniGPT(ModelConfig(12,8,8,2,2,0.)).eval()
ids = torch.tensor([[1,2,3,4]])
reference_trace = {}
with torch.no_grad():
    reference_logits,_ = model(ids,trace=reference_trace)
print('trace fields:',list(reference_trace))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace_logits(model, ids, use_positions=True)</code> دیکشنری با کلیدهای <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">combined</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">block_outputs</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">logits</code> برگرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">combined</code> جمع ورودی پیش از <bdi dir="ltr">Dropout</bdi>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">block_outputs</code> فهرست خروجی همهٔ بلوک‌ها و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">logits</code> خروجی <bdi dir="ltr">Head</bdi> پس از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">final_norm</code> باشد. از زیرلایه‌ها استفاده کنید، نه <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">model(ids)</code>.</p>
</div>

In [ ]:
def trace_logits(model, ids, use_positions=True):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = trace_logits(model,ids)
    if result is None: return False
    for tokens in (ids,torch.tensor([[4,3,2],[2,1,2]])):
        for positions in (False,True):
            actual = trace_logits(model,tokens,use_positions=positions)
            trace = {}; logits,_ = model(tokens,use_positions=positions,trace=trace)
            torch.testing.assert_close(actual['combined'],trace['combined_embedding'])
            assert len(actual['block_outputs']) == len(model.blocks)
            for a,b in zip(actual['block_outputs'],trace['layers']):
                torch.testing.assert_close(a,b['output'])
            torch.testing.assert_close(actual['logits'],logits)
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">use_positions</code> را روی همان مدل خاموش کنید. شکل‌ها ثابت می‌مانند، ولی مسیر ورودی تغییر می‌کند. این حذف کنترل‌شدهٔ موقعیت به‌تنهایی دربارهٔ کیفیت زبان یا بی‌اهمیت‌بودن ترتیب حکم نمی‌دهد.</p>
</div>

In [ ]:
with torch.no_grad():
    with_positions = model(ids,use_positions=True)[0]
    without_positions = model(ids,use_positions=False)[0]
print('same shape:',with_positions.shape == without_positions.shape)
print('position ablation change:',(with_positions-without_positions).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب <bdi dir="ltr">Attention</bdi> را روی <bdi dir="ltr">Embedding</bdi> خام اجرا می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">first_attention_input(model,ids)</code> فقط ورودی صحیحِ <bdi dir="ltr">Attention</bdi> بلوک اول را برگرداند: جمع موقعیت، <bdi dir="ltr">Dropout</bdi> ورودی و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">norm_1</code> را فراموش نکنید.</p>
</div>

In [ ]:
with torch.no_grad():
    wrong_input = model.token_embedding(ids)
    correct_input = model.blocks[0].norm_1(reference_trace['layers'][0]['input'])
print('raw embedding/input difference:',(wrong_input-correct_input).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def first_attention_input(model, ids):
    # TODO
    return None

In [ ]:
def test_repair():
    result = first_attention_input(model,ids)
    if result is None: return False
    torch.testing.assert_close(result,correct_input)
    for tokens in (torch.tensor([[2]]),torch.tensor([[2,3],[4,5]])):
        trace = {}; model(tokens,trace=trace)
        torch.testing.assert_close(first_attention_input(model,tokens),model.blocks[0].norm_1(trace['layers'][0]['input']))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">trace</code> از <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">forward</code> واقعی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> می‌آید و مسیر دستی با تک‌تک خروجی‌های بلوک تطبیق داده شد. مدل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">eval</code> و <bdi dir="ltr">Dropout</bdi> صفر است تا اختلاف تصادف را با اختلاف معماری اشتباه نگیریم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">اگر فقط <bdi dir="ltr">Logits</bdi> نهایی متفاوت بود، کدام مقایسهٔ میانی به شما کمک می‌کرد اولین نقطهٔ انحراف را پیدا کنید؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/45-trace.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>